# Model 1 — Inference Only
**Team11 | WU LLM Course SS26**

Runs a local Mistral-7B-Instruct model on all 643 Austrian tax law questions.
No retrieval, no fine-tuning — pure prompt-based inference.

**Platform:** Google Colab free T4 GPU  
**Runtime:** ~20–40 min for 643 questions on T4

In [ ]:
# Cell 1 — Install libraries
!pip install -q transformers accelerate bitsandbytes torch pandas

# imports — all here so a kernel restart doesn't break later cells
import os
import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.5 MB/s eta 0:00:00


In [ ]:
# Cell 2 — Upload dataset_clean.csv
# Run this cell, click "Choose Files", select dataset_clean.csv from your machine
from google.colab import files
uploaded = files.upload()  # upload dataset_clean.csv

Saving dataset_clean.csv to dataset_clean.csv


In [ ]:
# Cell 3 — Load model + tokenizer in 4-bit quantization
# 4-bit keeps memory usage low enough to fit on the free T4 GPU
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"  # freely accessible, no HF token needed

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto"  # automatically uses the GPU
)

print("Model loaded.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded.


In [ ]:
# Cell 4 — Load dataset and define shared prompt
import pandas as pd
import os

df = pd.read_csv("dataset_clean.csv")
print(f"Loaded {len(df)} questions.")

# Shared system prompt — same across all 3 models for fair comparison
# Citation format: § 26 Abs 1 Z 2 lit a EStG 1988  (no dot after Abs)
SYSTEM_PROMPT = (
    "Beantworte die folgende Frage zum österreichischen Steuerrecht auf Deutsch. "
    "Antworte in maximal 1–4 Sätzen. "
    "Nenne die einschlägige Rechtsnorm (z.B. § 7 Abs 1 KStG). "
    "Halluziniere keine Paragraphen. Wenn unklar, formuliere vorsichtig."
)

def build_prompt(question):
    # Format as Mistral instruction turn
    messages = [
        {"role": "user", "content": SYSTEM_PROMPT + "\n\n" + question}
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

Loaded 643 questions.


In [ ]:
# Cell 5 — Generate answers with checkpointing every 50 rows
# If the session crashes, re-run from this cell — it will resume from the checkpoint
CHECKPOINT = "checkpoint_model1.csv"
SAVE_EVERY = 50

# load checkpoint if it exists (resume after crash)
if os.path.exists(CHECKPOINT):
    done = pd.read_csv(CHECKPOINT)
    results = done.to_dict("records")
    done_ids = set(done["id"])
    print(f"Resuming from checkpoint: {len(results)} already done.")
else:
    results = []
    done_ids = set()

# loop through all questions
for i, row in df.iterrows():
    if row["id"] in done_ids:
        continue  # skip already completed

    prompt = build_prompt(row["prompt"])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # generate answer — keep it short (max 200 new tokens)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,        # greedy — deterministic output
            pad_token_id=tokenizer.eos_token_id
        )

    # decode only the newly generated tokens (skip the prompt)
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    results.append({"id": row["id"], "answer": answer})

    # save checkpoint every 50 rows
    if len(results) % SAVE_EVERY == 0:
        pd.DataFrame(results).to_csv(CHECKPOINT, index=False)
        print(f"  Checkpoint saved at row {len(results)}")

print(f"Done. {len(results)} answers generated.")

  Checkpoint saved at row 50
  Checkpoint saved at row 100
  Checkpoint saved at row 150
  Checkpoint saved at row 200
  Checkpoint saved at row 250
  Checkpoint saved at row 300
  Checkpoint saved at row 350
  Checkpoint saved at row 400
  Checkpoint saved at row 450
  Checkpoint saved at row 500
  Checkpoint saved at row 550
  Checkpoint saved at row 600


In [ ]:
# Cell 6 — Save final results to CSV
output_path = "model1_inference.csv"
pd.DataFrame(results).to_csv(output_path, index=False)
print(f"Saved to {output_path}")

# download the file to your machine
files.download(output_path)

Saved to model1_inference.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
# Cell 7 — Validate submission format
# Checks the output CSV before you move it to Team11/results/
ref = pd.read_csv("dataset_clean.csv")
sub = pd.read_csv(output_path)

assert list(sub.columns) == ["id", "answer"], "Wrong columns"
assert len(sub) == len(ref), f"Expected {len(ref)} rows, got {len(sub)}"
assert sub["id"].tolist() == ref["id"].tolist(), "IDs don't match"
assert sub["id"].nunique() == len(sub), "Duplicate IDs"
assert sub["answer"].notna().all(), "Null answers found"
assert (sub["answer"].str.strip() != "").all(), "Blank answers found"
top_freq = sub["answer"].value_counts().iloc[0]
assert top_freq < 20, f"Repeated answer {top_freq}x — possible crash"

print(f"OK — {output_path} is valid ({len(sub)} rows)")
print("\nSample answers:")
print(sub.head(3).to_string())

OK — model1_inference.csv is valid (643 rows)

Sample answers:
             id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                      answer
0  CORP-TAX-001                                                                                                                                                                                                                                                                                                                              Die Steuerbemessungsgrundlage für die Körperschaftsteuer in Österreich ist der w